# Data Understanding

---

## Objective

Understand dataset characteristics, transaction behavior, customer interactions, and business patterns prior to data cleaning.

---

## Business Perspective

Understanding customer purchasing behavior is essential to build an effective recommendation system.

Business stakeholders need to understand:

- Customer purchasing patterns.
- Product popularity.
- Transaction cancellations.
- Customer activity levels.

---

## Data Science Perspective

Data understanding helps identify hidden patterns, anomalies, and relationships among variables that influence recommendation quality.

This phase also provides evidence for data cleaning decisions.

---

## Methodology

The following analyses will be conducted:

1. Feature understanding.
2. Unique value exploration.
3. Transaction behavior analysis.
4. Product popularity analysis.
5. Customer activity analysis.
6. Country distribution analysis.
7. Cancellation analysis.
8. Return behavior analysis.

---

In [1]:
# ==================================================
# IMPORT LIBRARIES
# ==================================================

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==================================================
# LOAD DATASET
# ==================================================

PROJECT_ROOT = Path().resolve().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Online Retail.xlsx"

df = pd.read_excel(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (541909, 8)


# Feature Description

In [3]:
# ==================================================
# FEATURE OVERVIEW
# ==================================================

feature_description = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Unique Values": [df[col].nunique() for col in df.columns],
    "Missing Values": df.isnull().sum().values
})

display(feature_description)

,Feature,Data Type,Unique Values,Missing Values
InvoiceNo,InvoiceNo,object,25900,0
StockCode,StockCode,object,4070,0
Description,Description,object,4223,1454
Quantity,Quantity,int64,722,0
InvoiceDate,InvoiceDate,datetime64[us],23260,0
UnitPrice,UnitPrice,float64,1630,0
CustomerID,CustomerID,float64,4372,135080
Country,Country,str,38,0


In [4]:
# ==================================================
# UNIQUE VALUE EXPLORATION
# ==================================================

for col in df.columns:

    print("="*60)
    print(f"Column: {col}")
    print(f"Unique Values: {df[col].nunique()}")

    if df[col].nunique() <= 20:
        print(df[col].value_counts(dropna=False))

    print()

Column: InvoiceNo
Unique Values: 25900

Column: StockCode
Unique Values: 4070

Column: Description
Unique Values: 4223

Column: Quantity
Unique Values: 722

Column: InvoiceDate
Unique Values: 23260

Column: UnitPrice
Unique Values: 1630

Column: CustomerID
Unique Values: 4372

Column: Country
Unique Values: 38



In [5]:
# ==================================================
# CANCELLATION ANALYSIS
# ==================================================

cancelled = df["InvoiceNo"].astype(str).str.startswith("C").sum()

cancel_percentage = round(
    cancelled / len(df) * 100,
    2
)

print("Cancelled Transactions :", cancelled)
print("Cancellation Percentage :", cancel_percentage, "%")

Cancelled Transactions : 9288
Cancellation Percentage : 1.71 %


In [6]:
# ==================================================
# NEGATIVE QUANTITY ANALYSIS
# ==================================================

negative_quantity = (df["Quantity"] < 0).sum()

negative_percentage = round(
    negative_quantity / len(df) * 100,
    2
)

print("Negative Quantity Rows :", negative_quantity)
print("Percentage :", negative_percentage, "%")

Negative Quantity Rows : 10624
Percentage : 1.96 %


In [7]:
# ==================================================
# NEGATIVE PRICE ANALYSIS
# ==================================================

negative_price = (df["UnitPrice"] < 0).sum()

negative_price_percentage = round(
    negative_price / len(df) * 100,
    2
)

print("Negative Price Rows :", negative_price)
print("Percentage :", negative_price_percentage, "%")

Negative Price Rows : 2
Percentage : 0.0 %


In [8]:
# ==================================================
# TOP COUNTRIES
# ==================================================

country_dist = (
    df["Country"]
    .value_counts()
    .head(10)
)

display(country_dist)

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

In [9]:
# ==================================================
# TOP PRODUCTS
# ==================================================

top_products = (
    df["Description"]
    .value_counts()
    .head(10)
)

display(top_products)

Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
ASSORTED COLOUR BIRD ORNAMENT         1501
SET OF 3 CAKE TINS PANTRY DESIGN      1473
PACK OF 72 RETROSPOT CAKE CASES       1385
LUNCH BAG  BLACK SKULL.               1350
NATURAL SLATE HEART CHALKBOARD        1280
Name: count, dtype: int64

In [10]:
# ==================================================
# CUSTOMER ACTIVITY
# ==================================================

customer_activity = (
    df.groupby("CustomerID")
      .size()
      .sort_values(ascending=False)
)

display(customer_activity.head(10))

CustomerID
17841.0    7983
14911.0    5903
14096.0    5128
12748.0    4642
14606.0    2782
15311.0    2491
14646.0    2085
13089.0    1857
13263.0    1677
14298.0    1640
dtype: int64

In [11]:
# ==================================================
# COUNTRY SUMMARY
# ==================================================

country_summary = pd.DataFrame({

    "transactions":
    df.groupby("Country")["InvoiceNo"].nunique(),

    "customers":
    df.groupby("Country")["CustomerID"].nunique()

}).sort_values(
    by="transactions",
    ascending=False
)

display(country_summary.head(10))

,transactions,customers
Country,,
United Kingdom,23494,3950
Germany,603,95
France,461,87
EIRE,360,3
Belgium,119,25
Spain,105,31
Netherlands,101,9
Switzerland,74,21
Portugal,71,19


# Findings

1. A total of 9,288 transactions (1.71%) were identified as cancelled transactions.

2. The dataset contains 10,624 rows with negative quantities, indicating product returns, cancellations, or correction transactions.

3. Only two rows contain negative prices, suggesting rare correction or refund records.

4. The dataset is highly dominated by United Kingdom transactions, representing the majority of customer activity.

5. Product popularity is highly skewed, with several products appearing significantly more frequently than others.

6. Customer purchasing behavior is highly imbalanced, where a small number of customers generate a large volume of transactions.

7. The dataset exhibits characteristics commonly found in e-commerce recommendation systems, including sparse interactions and highly active customers.

8. Cancellation and return transactions may negatively affect recommendation quality if included without proper treatment.

---

# Decision

1. Cancelled transactions will be excluded from recommendation model training because they do not represent successful purchases.

2. Transactions with negative quantities will be removed during the data cleaning phase.

3. Records with negative prices will be removed because they do not represent valid sales transactions.

4. Recommendation models will be trained only on successful purchase transactions.

5. Product popularity information will be utilized for baseline recommendation modeling.

6. Customer activity distribution will be further explored during exploratory data analysis.

7. Additional feature engineering will be conducted after cleaning to create customer-product interaction matrices.